# mse-reconstruction-loss — faded example 3: Compute MSE with sum reduction and verify it equals mean * num_elements

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `mse-reconstruction-loss`. Running the beacon reports progress on the `Generative: MSE reconstruction loss` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: MSE reconstruction loss` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`mse-reconstruction-loss`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "mse-reconstruction-loss"
DD_SUBTOPIC = "Generative: MSE reconstruction loss"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The `reduction` argument to `F.mse_loss` controls whether the squared errors are summed (`'sum'`) or averaged (`'mean'`) over all elements. Some VAE implementations use `reduction='sum'` for the reconstruction term so that the overall loss scales with batch size consistently with a per-sample KL divergence term. Understanding the relationship between sum and mean reduction prevents loss-scale mismatches.

## Faded exercise 3

Implement `mse_sum_and_mean(pred, target)` that returns a tuple `(loss_sum, loss_mean)` where:
- `loss_sum  = F.mse_loss(pred, target, reduction='sum')`
- `loss_mean = F.mse_loss(pred, target, reduction='mean')`

Your task: **fill in both loss computations and return them as a tuple**.

**Fill in:** Computing F.mse_loss with reduction='sum' and reduction='mean' and returning both as a tuple (loss_sum, loss_mean).

In [ ]:
import torch
import torch.nn.functional as F

def mse_sum_and_mean(pred: torch.Tensor, target: torch.Tensor):
    raise NotImplementedError()  # TODO: Computing F.mse_loss with reduction='sum' and reduction='mean' and returning both as a tuple (loss_sum, loss_mean).

def _test():
    import torch, torch.nn.functional as F
    torch.manual_seed(0)
    B, C, H, W = 3, 2, 5, 5
    pred   = torch.randn(B, C, H, W)
    target = torch.randn(B, C, H, W)
    loss_sum, loss_mean = mse_sum_and_mean(pred, target)
    n = B * C * H * W
    assert abs(loss_sum.item() - loss_mean.item() * n) < 1e-4, (
        f"sum={loss_sum.item():.4f}, mean*n={loss_mean.item()*n:.4f}")
    assert loss_sum.item() > 0


def _test():
    import torch, torch.nn.functional as F
    torch.manual_seed(0)
    B, C, H, W = 3, 2, 5, 5
    pred   = torch.randn(B, C, H, W)
    target = torch.randn(B, C, H, W)
    loss_sum, loss_mean = mse_sum_and_mean(pred, target)
    n = B * C * H * W
    # sum should equal mean * total_elements
    assert abs(loss_sum.item() - loss_mean.item() * n) < 1e-3, (
        f"sum={loss_sum.item():.5f} != mean*n={loss_mean.item()*n:.5f}")
    # both positive
    assert loss_sum.item() > 0
    assert loss_mean.item() > 0
    # sum > mean (unless n==1)
    assert loss_sum.item() > loss_mean.item()
    # independent ground truth for mean
    ref_mean = F.mse_loss(pred, target)
    assert abs(loss_mean.item() - ref_mean.item()) < 1e-6


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch
import torch.nn.functional as F

def mse_sum_and_mean(pred: torch.Tensor, target: torch.Tensor):
    loss_sum  = F.mse_loss(pred, target, reduction='sum')
    loss_mean = F.mse_loss(pred, target, reduction='mean')
    return loss_sum, loss_mean

def _test():
    import torch, torch.nn.functional as F
    torch.manual_seed(0)
    B, C, H, W = 3, 2, 5, 5
    pred   = torch.randn(B, C, H, W)
    target = torch.randn(B, C, H, W)
    loss_sum, loss_mean = mse_sum_and_mean(pred, target)
    n = B * C * H * W
    assert abs(loss_sum.item() - loss_mean.item() * n) < 1e-4
    assert loss_sum.item() > 0
```
</details>